<a href="https://colab.research.google.com/github/keszkowskim/BAN5600-800-Homework/blob/main/Homework4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Reusable DataTable function**

In [13]:
from IPython.display import display, HTML
import json

def render_datatable(table_id, title, columns, data):
    columns_js = json.dumps([{"title": col} for col in columns])
    data_js = json.dumps(data)

    html = f"""
    <h3>{title}</h3>

    <!-- DataTables + Buttons CSS -->
    <link rel="stylesheet" href="https://cdn.datatables.net/1.13.6/css/jquery.dataTables.min.css">
    <link rel="stylesheet" href="https://cdn.datatables.net/buttons/2.4.1/css/buttons.dataTables.min.css">

    <!-- jQuery + DataTables + Buttons JS -->
    <script src="https://code.jquery.com/jquery-3.7.0.min.js"></script>
    <script src="https://cdn.datatables.net/1.13.6/js/jquery.dataTables.min.js"></script>
    <script src="https://cdn.datatables.net/buttons/2.4.1/js/dataTables.buttons.min.js"></script>
    <script src="https://cdn.datatables.net/buttons/2.4.1/js/buttons.html5.min.js"></script>
    <script src="https://cdn.datatables.net/buttons/2.4.1/js/buttons.print.min.js"></script>

    <!-- Required for Excel export -->
    <script src="https://cdnjs.cloudflare.com/ajax/libs/jszip/3.10.1/jszip.min.js"></script>

    <table id="" class="display {table_id}" style="width:100%"></table>

    <script>
    $(document).ready(function() {{

        if ($.fn.DataTable.isDataTable(".{table_id}")) {{
            $('.{table_id}').DataTable().destroy();
            $('.{table_id}').empty();
        }}

        $('.{table_id}').DataTable({{
            data: {data_js},
            columns: {columns_js},
            pageLength: 10,
            dom: 'Bfrtip',  // <-- THIS ENABLES BUTTONS

            buttons: [
                'copy',
                'csv',
                'excel',
                'print'
            ],

            createdRow: function(row, data) {{
                if (data.includes("Urgent")) {{
                    $(row).css("background-color", "#ffdddd");
                }}
                if (data.includes("Overdue")) {{
                    $(row).css("background-color", "#fff3cd");
                }}
                if (data.includes("Missing")) {{
                    $(row).css("background-color", "#fde2e2");
                }}
            }}
        }});
    }});
    </script>
    """

    display(HTML(html))

**1 - Problem 2 — Low Inventory and Reorder Needs**

In [14]:
inventory = [
    ["A101", "Pens", "Office Supplies", 12, 20, "Staples"],
    ["A102", "Paper", "Office Supplies", 50, 25, "OfficeMax"],
    ["A103", "Coffee Cups", "Breakroom", 8, 15, "Costco"],
    ["A104", "Markers", "Office Supplies", 3, 10, "Staples"],
    ["A105", "Napkins", "Breakroom", 30, 30, "Costco"]
]

low_stock_data = []
urgent_items = []
index = 0

while index < len(inventory):
    item = inventory[index]
    item_name = item[1]
    current_stock = item[3]
    reorder_level = item[4]
    supplier = item[5]

    if current_stock > reorder_level:
        index += 1
        continue

    short_by = reorder_level - current_stock
    status = "Urgent" if current_stock <= 5 else "Normal"

    low_stock_data.append([item_name, short_by, supplier, status])

    if status == "Urgent":
        urgent_items.append(item_name)

    index += 1

for row in low_stock_data:
    if row[0] == "Markers":
        break

print("Total low stock items:", len(low_stock_data))
print("Urgent items:", urgent_items)

render_datatable(
    "inventoryTable",
    "Problem 2: Low Inventory Report",
    ["Item", "Short By", "Supplier", "Status"],
    low_stock_data
)

Total low stock items: 4
Urgent items: ['Markers']


**2 - Problem 3 — Overdue Customer Payments**

In [15]:
from datetime import datetime
#from datetime import date

invoices = [
    ["INV001", "ABC Company", "2026-03-01", "2026-03-20", 1200, "Unpaid"],
    ["INV002", "XYZ Inc", "2026-03-05", "2026-04-25", 850, "Unpaid"],
    ["INV003", "Bright Media", "2026-02-15", "2026-03-01", 640, "Paid"],
    ["INV004", "Delta Group", "2026-03-10", "2026-03-30", 1500, "Unpaid"],
    ["INV005", "Nova LLC", "2026-03-12", "2026-04-10", 400, "Unpaid"]
]

#today = date.today().strftime("%Y-%m-%d")
#print(formatted_date)

#today = datetime.strptime("2026-04-19", "%Y-%m-%d")

today = datetime.today()

overdue_data = []
total_overdue = 0
i = 0

while i < len(invoices):
    invoice = invoices[i]
    invoice_id = invoice[0]
    customer = invoice[1]
    due_date = datetime.strptime(invoice[3], "%Y-%m-%d")
    amount = invoice[4]
    payment_status = invoice[5]

    if payment_status == "Paid":
        i += 1
        continue

    days_overdue = (today - due_date).days

    if days_overdue > 0:
        status = "Overdue"
        overdue_data.append([invoice_id, customer, amount, days_overdue, status])
        total_overdue += amount

    i += 1

for row in overdue_data:
    if row[3] > 25:
        print("Immediate follow-up needed for:", row[1])
        break

print("Total overdue balance:", total_overdue)

render_datatable(
    "overdueTable",
    "Problem 3: Overdue Payments Report",
    ["Invoice ID", "Customer", "Amount", "Days Overdue", "Status"],
    overdue_data
)



Immediate follow-up needed for: ABC Company
Total overdue balance: 3100


**3 - Problem 4 — Cleaning and Standardizing Messy Business Data**

In [23]:
from datetime import datetime
from dateutil import parser

records = [
    ["  john smith ", "01/15/2025", "east ", "1200", "JOHN@email.com "],
    ["mary jones", "2025-02-10", " West", "", "mary@email.com"],
    ["MARY JONES", "2025-02-10", "west", "", "mary@email.com"],
    [" peter pan", "3/5/2025", "NORTH ", "950", " "],
    ["lisa ray ", "2025/04/01", "south", "1100", "lisa@email.com"]
]

cleaned_data = []
seen = []
index = 0

while index < len(records):
    row = records[index]

    name = row[0].strip().title()
    date_object = parser.parse(row[1])
    # Format the datetime object to 'm/d/Y'
    join_date =  date_object.strftime("%#m/%#d/%Y")
    region = row[2].strip().title()
    sales = row[3]
    email = row[4].strip().lower()

    if sales == "":
        sales = "0"

    if email == "":
        index += 1
        continue

    cleaned_row = [name, join_date, region, float(sales), email, "Clean"]

    if cleaned_row in seen:
        index += 1
        continue

    seen.append(cleaned_row)
    cleaned_data.append(cleaned_row)
    index += 1

for row in cleaned_data:
    if row[2] == "North":
        print("North region record found.")
        break

print("Records before cleaning:", len(records))
print("Records after cleaning:", len(cleaned_data))

render_datatable(
    "cleaningTable",
    "Problem 4: Cleaned Business Data",
    ["Customer Name", "Join Date", "Region", "Sales Amount", "Email", "Status"],
    cleaned_data
)

Records before cleaning: 5
Records after cleaning: 3


**4 - Problem 5 — Merging Reports from Multiple Departments**

In [24]:
customers = [
    [1, "John Smith", "East"],
    [2, "Mary Jones", "West"],
    [3, "Peter Pan", "North"],
    [4, "Lisa Ray", "South"]
]

orders = [
    [1, "ORD1001", "2026-04-01", 300],
    [2, "ORD1002", "2026-04-05", 450],
    [2, "ORD1003", "2026-04-07", 200],
    [4, "ORD1004", "2026-04-10", 150],
    [5, "ORD1005", "2026-04-12", 500]
]

merged_data = []
matched_customers = []
i = 0

while i < len(customers):
    customer = customers[i]
    customer_id = customer[0]
    customer_name = customer[1]
    region = customer[2]
    found_order = False

    for order in orders:
        if order[0] != customer_id:
            continue

        merged_data.append([customer_name, region, order[1], order[2], order[3], "Matched"])
        found_order = True
        matched_customers.append(customer_name)

    if found_order == False:
        merged_data.append([customer_name, region, "No Order", "N/A", 0, "Missing"])

    i += 1

for row in merged_data:
    if row[2] == "ORD1003":
        print("Customer with multiple orders found.")
        break

print("Merged rows:", len(merged_data))
print("Customers with purchases:", matched_customers)

render_datatable(
    "mergeTable",
    "Problem 5: Merged Department Report",
    ["Customer Name", "Region", "Order ID", "Order Date", "Order Amount", "Status"],
    merged_data
)

Customer with multiple orders found.
Merged rows: 5
Customers with purchases: ['John Smith', 'Mary Jones', 'Mary Jones', 'Lisa Ray']


**Problem 5: Automating Lead Qualification for WealthSnap.io (A Kesz1 Tech Company)**

For my custom problem, I chose to focus on automating lead qualification for my company, WealthSnap.io. Currently, when potential clients submit a form or inquiry, someone has to manually review their information to determine if they are a good fit for the service. This process can be time-consuming and inconsistent, especially as the number of leads increases.

The objective of this automation would be to analyze key data points such as income level, investable assets, financial goals, and urgency of need, and then assign a lead score. Based on that score, the system could categorize leads as high, medium, or low priority and route them accordingly. High-priority leads could be flagged for immediate follow-up, while lower-priority leads could be nurtured through automated email sequences. This type of automation would improve efficiency, reduce manual work, and help ensure that the most valuable leads are prioritized, ultimately increasing conversion rates and overall business performance.